# Static dashboard


In [ ]:
# general
import geopandas as gpd
import pandas as pd
import numpy as np
import os

# plotting
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go
from   plotly.subplots import make_subplots

from utils.config import find_repo_root, get_data_root

wd = find_repo_root()
os.chdir(wd)

## Data

In [ ]:
# shapes/polygons
AndeanGC_data     = pd.read_csv('dataset/AndeanGC_data_1950_2024.csv', index_col=0, parse_dates=['date'])
AndeanGC_data_yearly = AndeanGC_data.notna().sum(axis = 1).resample('YS').mean()

AndeanGC_metadata = pd.read_csv('dataset/AndeanGC_metadata.csv', index_col=0)
AndeanGC_shape    = gpd.read_file('dataset/AndeanGC_shape.gpkg').set_index("gauge_id")
AndeanGC_shape    = AndeanGC_shape.sort_values(['basin_area', 'gauge_lat'], ascending=[False, True])

## Plot

In [ ]:
# Create subplot structure
fig = make_subplots(
    rows=2, cols=3,
    vertical_spacing=0.08, 
    horizontal_spacing=0.06, 
    column_widths=[0.28, 0.35, 0.35],
    specs=[[{"type": "scattergeo", "rowspan": 2}, {"type": "xy", "colspan": 2}, None],
           [None, {"type": "xy"},  {"type": "xy"}]]
)

# Plot 1: Glacier area (scatter geo map)
fig.add_trace(
    go.Scattergeo(
        lon=AndeanGC_shape['basin_cenlon'],
        lat=AndeanGC_shape['basin_cenlat'],
        mode='markers',
        marker=dict(
            size=scaler.fit_transform(AndeanGC_shape[['basin_area']]).flatten(),
            opacity=0.7,
            color=AndeanGC_shape['glacier_area_RGI60'],
            colorscale="Viridis_r",
            cmin=0,
            cmax=20,
            colorbar=dict(
                len=0.4, x=0.01, y=0.63,
                title="Glacier<br>area (km²)",
                thickness=15, tickwidth=1,
                bgcolor="rgba(255,255,255,0)"
            ),
            line=dict(width=0.5, color='grey')
        ),
        showlegend=False
    ),
    row=1, col=1
)

# Plot 2: Data availability (bar chart)
fig.add_trace(
    go.Bar(
        x=AndeanGC_data_yearly.index.year,
        y=AndeanGC_data_yearly.values,
        marker_color="#3979a7",
        name='Data availability',
        showlegend=False
    ),
    row=1, col=2
)

# Plot 3: Years with data (histogram)
fig.add_trace(
    go.Histogram(
        x=AndeanGC_metadata.days_w_data_qc/365,
        marker_color="#3979a7",
        name='Data availability',
        showlegend=False,
        nbinsx=20
    ),
    row=2, col=2
)

# Plot 4: Basin area (log-scale histogram)
fig.add_trace(
    go.Histogram(
        x=np.log10(AndeanGC_metadata.basin_area),
        marker=dict(color="#3979a7"),
        name='Data availability',
        showlegend=False,
        nbinsx=20
    ),
    row=2, col=3
)

# Add country labels
fig.add_trace(
    go.Scattergeo(
        lon=[-69.0, -64.0, -73.0, -64.0],
        lat=[-23.5, -27.0, -11.0, -18.0],
        text=["Chile", "Argentina", "Peru", "Bolivia"],
        mode="text",
        textfont=dict(size=12, color="black"),
        showlegend=False
    ),
    row=1, col=1
)

# Update axes labels and ranges
fig.add_annotation(text=f'Total stations: {len(AndeanGC_metadata)}', 
                    x=1958, y=233, showarrow=False, font=dict(size=16), row=1, col=2)

fig.add_annotation(text=f'Median: {np.median(AndeanGC_metadata.days_w_data_qc/365):.0f} years',
                    x=50, y=37, showarrow=False, font=dict(size=16), row=2, col=2)

fig.update_xaxes(griddash="dot", ticks="outside", row=1, col=2)
fig.update_yaxes(title_text='Gauges with data (N)', range=[0, 250], griddash="dot", ticks="outside", row=1, col=2)

fig.update_xaxes(title_text='Years with data', row=2, col=2)
fig.update_yaxes(title_text='Gauges (N)', range=[0, 40], row=2, col=2)

fig.update_xaxes(
    title_text='Basin area (km<sup>2</sup>)',
    tickvals=[2, 3, 4],
    ticktext=["10<sup>2</sup>", "10<sup>3</sup>", "10<sup>4</sup>"],
    row=2, col=3
)
fig.update_yaxes(title_text='Gauges (N)', range=[0, 40], row=2, col=3)

# Apply geographic settings and global axis styling
water = "#ecf1f8"
fig.update_geos(
    lonaxis_range=[-80, -59],
    lataxis_range=[-56, -7],
    showframe=True, 
    framecolor="#ecf1f8", bgcolor="rgba(210, 226, 230, 0.7)",
    landcolor="white", showland=True, resolution=50,
    showlakes=True, showocean = True, lakecolor=water, oceancolor=water,
    showcoastlines=True, coastlinecolor="rgba(0,0,0,0)",
    showcountries=True, countrycolor="rgba(0,0,0,0.2)", countrywidth =0.2,
    row=1, col=1
)
fig.update_xaxes(griddash="dot", ticks="outside", title_standoff=5)
fig.update_yaxes(griddash="dot", ticks="outside", title_standoff=5)

# Update layout
fig.update_layout(
    autosize=False,
    width=1000,
    height=600,
    bargap=0.1,
    plot_bgcolor=water,
    margin=dict(l=10, r=10, b=10, t=10)
)

fig.show()
fig.write_image("figures/figure01_map_data.png", width=1000, height=600, scale=4)